## PARIS inverse modelling results

This notebook contains options to plot and compare results from different inverse models. These variables can all be investigated:

- Posterior and prior country fluxes, total from all sectors
- Posterior, prior and observed modelled total mole fractions
- Posterior and prior modelled baseline mole fractions
- Posterior and prior spatial fluxes

Future updates may include:

- Sector-level emissions
- Comparison between each model's country/region definition

### Notebook setup:

1. Edit the `data_dir` to point towards where the model output is.

2. Update the `model_filenames` dictionary to point towards the experiment/model you want to plot. Update the `model_labels` and `model_colors` dictionaries, these change how each model is labelled in the plots.

3. Run the cell below, before running any of the plotting code.

In [ ]:
%load_ext autoreload
%autoreload 2
import logging
logging.basicConfig(level=logging.WARNING)

from fluxy.config import set_print_settings
from fluxy.config import set_model_colors
from fluxy.config import set_model_labels
from fluxy.io import read_config_files, read_model_output, read_flux_total_fgases
from fluxy.operators.select import slice_flux
from fluxy.plots.mf_timeseries import plot_sites_timeseries
from fluxy.plots.mf_timeseries import plot_mf_timeseries
from fluxy.operators.mf import compute_mf_difference

### Path to results directory 
data_dir = 'path/to/data'

### Set presentation_mode to True for bigger fonts
presentation_mode = False

### Group the models of interest in meaningful experiment names 
experiments = {'std_run'       :['InTEM_NAME_EDGAR_std','ELRIS_NAME_EDGAR_std','RHIME_NAME_EDGAR_std'], # Standard run 
               'obs_selection' :['intem_name_edgar','intem_name_edgar_allobs',              # Effect of obs selection
                                 'elris_name_edgar','elris_name_edgar_allobs',
                                 'rhime_name_edgar','rhime_name_edgar_allobs'],
               'elris_edgar'    :['elris_name_edgar','elris_name_edgar_allobs',              # Effect of transport model and obs selection (ELRIS only)
                                  'elris_flex_edgar','elris_flex_edgar_allobs'],
               'wetcharts'      :['intem_name_wetcharts_31sites','elris_name_wetcharts_31sites',
                                  'rhime_name_wetcharts_31sites']
              }

###################################

config_data = read_config_files()
annotate_coords = set_print_settings(presentation_mode)

### 1. Timeseries of country/region fluxes

##### Edit and run this cell to choose inputs:

In [ ]:
###################################
### edit variables in this block
specie = 'hfc134a' #options for individual species, or 'all_hfc' or 'all_pfc'
models = experiments['std_run'] #if using 'all_hfc' or 'all_pfc' use basic model names (e.g. ['InTEM','ELRIS','RHIME']) and standard filenames will be used for each species
regions = ['GERMANY','UK','BENELUX','NW_EU2']
period = 'yearly' #inversion period, must be a string or a list of the same length as models, e.g. ['monthly','yearly']
country_flux_units_print = 'Gg yr-1'
start_date = '2018-01-01' #inclusive. Option to set as list of dates, e.g. ['2018-01-01','2019-01-01'] which is required for total fgases if one model is missing obs for a year
end_date = '2024-01-01' #not inclusive. Option to set as list of dates, e.g. ['2023-01-01','2022-01-01'] which is required for total fgases if one model is missing obs for a year
get_labels_from_file = False # If False, uses default labels. If True, uses labels from config file.
###################################

ds_all_flux_scaled = {}

if 'all' in specie:
    ds_all_flux_scaled = read_flux_total_fgases(data_dir,specie,models,config_data,
                                                     regions,start_date,end_date,
                                                     period=period)
else:
    ds_all_flux = read_model_output(data_dir,"flux",specie,models,config_data,period=period)

    for m in models:
        ds_all_flux_scaled[m] = slice_flux({m:ds_all_flux[m]},config_data,start_date,end_date,specie=specie,country_flux_units_print=country_flux_units_print)[m]

model_colors = set_model_colors(models)
model_labels = set_model_labels(models,config_data,get_labels_from_file)

#### Timeseries of country fluxes:

In [ ]:
%matplotlib inline
###################################
### edit variables in this block
from fluxy.plots.flux_timeseries import plot_country_flux


plot_inventory = False
inventory_years = None #If None, plots most recent. Or can choose list of years: ['2022','2023']
fix_y_axes = False #if True: all y axis limits are the same, if False: each y axis is relative to the data
                     #if a list of floats (e.g. [0,0.1]) applies these limit to all axes
add_prior = True #if True: plots prior as dashed lines
add_prior_unc = False #if True: plots prior uncertainty as shaded area
set_global_leg = True #If True, plots one single legend instead of one legend per subplot.
country_codes_as_titles = False #If True, lists 3-letter country codes under region names in subplot titles. Set to None for no title.
plot_separate = True #If True, includes all model results as separate lines (or insert a list of boolean of the same length as models to specify which models to plot)
plot_combined = False #If True, combined results, averaged from all models (or insert a list of boolean of the same length as models to specify which models to combine)
resample = None #If None, no resample is done. Else resample the data to the given period (options 'year' and 'season' for yearly and seasonal averages)
resample_uncert_correlation = False #If True, uses mean uncertainty during resampling, if False, recalculates uncertainty assuming no correlation.
plot_resample_and_original = False #If True, plots both the resampled and original data
period_override = None #use to override standard inversion periods, must be a list the same length as models, e.g. ['monthly','yearly']
annex_mode = False #If True, replace the labels with more concise versions for NID Annexes.
rolling_mean = False ##If True, calculates a rolling mean of the data (insert a list of boolean of the same length as models to specify the models to smooth)
###################################

fig = plot_country_flux(ds_all_flux_scaled,
                        specie,
                        regions,
                        config_data['species_info'],
                        model_colors,
                        model_labels,
                        start_date,
                        end_date,
                        annex_mode,
                        plot_inventory,
                        inventory_years,
                        data_dir,
                        fix_y_axes,
                        add_prior,
                        add_prior_unc,
                        set_global_leg,
                        country_codes_as_titles=country_codes_as_titles,
                        plot_separate=plot_separate,
                        plot_combined=plot_combined,
                        resample=resample,
                        resample_uncert_correlation=resample_uncert_correlation,
                        plot_resample_and_original=plot_resample_and_original,
                        rolling_mean=rolling_mean,
                        return_res=True)

Save plot here:

In [ ]:
# output_path = '/project/paris/users/AlexDanjou/images/flux_country.png'

# fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

### 2. Modelled and observed mole fractions and/or baselines

##### Edit and run this cell to choose inputs and read in the data:

In [ ]:
from fluxy.io import read_model_output
from fluxy.operators.select import slice_mf

###################################
### edit variables in this block
specie = 'hfc134a' 
site = 'MHD'
models = experiments['std_run']
period = 'yearly'  #inversion period, must be a string or a list of the same length as models, e.g. ['monthly','yearly']
mf_units_print = 'ppt'
start_date = '2018-01-01' #inclusive
end_date = '2019-01-01'   #not inclusive
baseline_site = None #'MHD', 'JFJ' or 'CMN'. If None, does not mask by baseline time
get_labels_from_file = False # If False, uses default labels. If True, uses labels from config file.
###################################

###################################
### options for variables to include in the functions below 
# Yobs                - total observed mole fraction
# uYobs_repeatability - observed repeatability mole fraction uncertainty
# uYobs_variability   - observed variability mole fraction uncertainty
# uYmod               - model mole fraction uncertainty
# uYtotal             - total mole fraction uncertainty
# qYapriori
# qYapost
# Yapriori            - prior total mole fraction
# Yapost              - posterior total mole fraction
# YaprioriBC          - prior baseline
# YapostBC            - posterior baseline
# Yapriori_bias       - prior bias added to site
# Yapost_bias         - posterior bias added to site
# YaprioriOUTER       - prior mole fractions only from outer regions
# YapostOUTER         - posterior mole fractions only from outer regions
###################################

ds_all_mf = read_model_output(data_dir,'concentration',specie,models,config_data,period=period)

ds_all_mf_sliced = slice_mf(ds_all_mf.copy(),start_date,end_date,site,baseline_site=baseline_site,
                              data_dir=data_dir,
                              mf_units_print=mf_units_print)

model_colors = set_model_colors(models)
model_labels = set_model_labels(models,config_data,get_labels_from_file)

#### Plot timestamps with data at each site:

In [ ]:
fig = plot_sites_timeseries(ds_all_mf,'Yapost',start_date,end_date,model_colors,model_labels)

Save plot:

In [ ]:
# output_path = '/project/paris/users/AlexDanjou/images/sites.png'

# fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

#### Timeseries plot, separated by model:

In [ ]:
# Variables and respective uncertainties to plot
include = {'Yobs'   : None,
           'Yapost' : 'qYapost'
           }

fig = plot_mf_timeseries(ds_all_mf_sliced,specie,site,
                                    model_colors,model_labels,config_data,annotate_coords,presentation_mode,
                                    plot_type='separate',
                                    include=include,
                                    diff_include=['Yapost'],
                                    y_lim=None)

###################################
### Instructions for timeseries plotting: 
# - Add variables that you want to plot in "include"
# - To plot the histogram of the variables in "include", leave "diff_include" empty
# - To plot the histogram of Obs-variable, add the desired variable to be subtracted in "diff_include" (default value of "diff_include" = Yapost)
# - To choose y axis limits set y_lim=[min_value,max_value]
###################################

Save plot:

In [ ]:
# output_path = '/project/paris/users/AlexDanjou/images/Yapost_timeseries.png'

# fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

#### Timeseries plot, all models together:

In [ ]:
# Variables and respective uncertainties to plot
include = {'Yapost' : 'qYapost'}

fig = plot_mf_timeseries(ds_all_mf_sliced,specie,site,
                                    model_colors,model_labels,config_data,annotate_coords,presentation_mode,
                                    plot_type='together',
                                    include=include,
                                    diff_include=['Yapost'],
                                    y_lim=None)

Save plot:

In [ ]:
# output_path = '/project/paris/users/AlexDanjou/images/Yapost_allmodels.png'

# fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

#### Absolute differences between two models:

In [ ]:
# Models to subtract
models_to_subtract = models[:2] # list with 2 elements

# Variables to plot (set all dict values to None)
include = {'Yobs': None}

ds_diff = compute_mf_difference(ds_all_mf_sliced.copy(), models_to_subtract)

fig = plot_mf_timeseries(ds_diff,specie,site,
                        model_colors,model_labels,config_data,annotate_coords,presentation_mode,
                        plot_type='diff',
                        include=include,
                        diff_include=None,
                        y_lim=None)

Save plot:

In [ ]:
# output_path = '/project/paris/users/AlexDanjou/images/Yapost_diff.png'

# fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

#### Calculate and plot stats for the fit to obs:

In [ ]:
from fluxy.operators.mf import stats_mf
from fluxy.plots.mf_stats import plot_stats_mf

###################################
### edit variables in this block
stats_to_plot = ['pearson','nrmse','rmse']

### Implemented statistics 
# pearson - Pearson correlation coefficient
# nrmse   - Normalised RMSE
# rmse    - RMSE
# std     - Standard deviation
###################################

ds_all_allsites = slice_mf(ds_all_mf.copy(),start_date,end_date,site=None,
                              baseline_site=baseline_site,
                              data_dir=data_dir,
                              mf_units_print=mf_units_print)

stats = stats_mf(ds_all_allsites)

fig = plot_stats_mf(stats,stats_to_plot,specie,
                    model_colors,model_labels,config_data,
                    start_date=start_date,end_date=end_date)

Save plot:

In [ ]:
#output_path = None

#fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

### 3. Spatial maps of country/region fluxes

In [ ]:
###################################
specie = 'hfc134a'
start_date = '2018-01-01' 
end_date = '2020-01-01'
models = experiments['std_run']
period = 'yearly' #inversion period, must be a string or a list of the same length as models, e.g. ['monthly','yearly']
flux_units_print = 'kg km-2 yr-1'
get_labels_from_file = False # If False, uses default labels. If True, uses labels from config file.
###################################

ds_all_flux = read_model_output(data_dir,"flux",specie,models,config_data,period=period)

ds_all_flux_scaled = {}
for m in models:
    ds_all_flux_scaled[m] = slice_flux({m:ds_all_flux[m]},config_data,start_date,end_date,specie=specie,flux_units_print=flux_units_print)[m]

model_labels = set_model_labels(models,config_data,get_labels_from_file)

##### Prior and posterior fluxes comparison across models

In [ ]:
###################################
from fluxy.plots.flux_map import plot_flux_map

region = 'NW_EU'
add_sites = True
add_markers = ['paris'] # Add a marker at given locations, options for 'paris', 'london', ... or any value [lon, lat]
season = None # If specified plot the seasonal mean (only valable for monthly data), options for 'DJF', 'MAM', 'JJA' and 'SON'
set_fluxlim = 'auto' # Set flux colorbar limits: e.g.[0,10]|'default'|'auto'
                        # 'default' = species_info default values and 'auto' = 99th percentile of flux_total_posterior
plot_inversion_grid_flux = True # If True, plot fluxes at inversion's resolution, if False, plot fluxes at prior's resolution.
###################################

fig = plot_flux_map(ds_all_flux_scaled, specie, 
                    region, config_data,
                    period_override=period_override, season=season,
                    add_sites=add_sites, add_markers=add_markers,
                    set_fluxlim=set_fluxlim, plot_inversion_grid_flux=plot_inversion_grid_flux)

Save plot:

In [ ]:
#output_path = None
#fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

##### Flux comparison between two models

In [ ]:
###################################
from fluxy.plots.flux_map import plot_flux_map_model_comparison

var = 'flux_total_posterior_inversion_grid'
model_1 = 'intem_name_edgar'
model_2 = 'elris_name_edgar'
###################################

fig = plot_flux_map_model_comparison(ds_all_flux_scaled, var, 
                                     model_1, model_2,
                                     specie, region, 
                                     config_data, presentation_mode,
                                     period_override=period_override, season=season,
                                     add_sites=add_sites, add_markers=add_markers,
                                     set_fluxlim=set_fluxlim)

Save plot:

In [ ]:
#output_path = None
#fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

##### Plot prior, posterior or difference fluxes per time interval:

In [ ]:
###################################
### edit variables in this block
from fluxy.plots.flux_map import plot_spatial_flux_per_timestamp


plot_area = 'CWEU'       #options for: UK, FRANCE, GERMANY, ITALY, SWITZERLAND, NETHERLANDS, IRELAND, HUNGARY, NORWAY, NWEU, CWEU, EUROPE
cmap = 'viridis'         #flux colour (e.g. 'viridis', 'coolwarm')
c_border = 'floralwhite' #country boundary colour (e.g. 'floralwhite', 'dimgrey')
chop_by = 'year'  #time unit of the averaging period (options for 'year', 'month' and 'season')
                  #alternatively, a list of starting dates can be provided (format '2018-01-01')
dt = 1  #number of time steps (in chop_by units) to use in the averaging
         #if chop_by='season': set dt to a list of lists with the months to use in the average (e.g. dt=[[1,2],[10,11]], will average over Jan-Feb and Oct-Nov)
var = 'flux_total_posterior'  #variable to be plotted. Options for: 'flux_total_posterior', 'flux_total_prior', 'posterior_prior_diff', 'posterior_mean_diff'
plot_combined = False
plot_site_locations = True    #If True, adds markers to plots at locations of obs sites
plot_point_markers = ['paris','london'] #plot a marker at these locations, options for 'paris', 'london', 'nw_england' (PFC-218 source) or any value [lon,lat]
plot_point_markers = None #plot a marker at these locations, options for 'paris', 'london', 'nw_england' (PFC-218 source) or any value [lon,lat]
set_fluxlim = 'auto' # Set flux colorbar limits: e.g.[0,10]|'default'|'auto'
                        # 'default' = species_info default values and 'auto' = 99th percentile of var
set_fluxlim_percentile = None # Percentile to use when setting the colorbar limits with 'auto' option.
annex_mode = False #If True, replace the labels with more concise versions for NID Annexes.
plot_inversion_grid_flux = False #If True, plots fluxes at resolution of inversion, if False, plots at the resolution of the prior
###################################

fig = plot_spatial_flux_per_timestamp(ds_all_flux_scaled,species,plot_area,end_date,s_data,m_data,
                                            cmap=cmap,c_border=c_border,var=var,
                                            plot_combined=plot_combined,annex_mode=annex_mode,chop_by=chop_by,dt=dt,period_override=period_override,
                                            plot_site_locations=plot_site_locations,
                                            plot_point_markers=plot_point_markers,set_fluxlim = set_fluxlim,
                                            set_fluxlim_percentile=set_fluxlim_percentile,plot_inversion_grid_flux=plot_inversion_grid_flux)

Save plot:

In [ ]:
#output_path = None
#fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)